# Example inference (image and video)

This notebook runs pose estimation on an image and a video: it shows **side-by-side** original vs annotated for an image, and exports **5-second GIFs** of the original and annotated video.


In [ ]:
# Load all necessary libraries
import sys
from pathlib import Path

# Add project root so we can use configs or scripts if needed
_project_root = Path.cwd().resolve()
if _project_root.name == "notebooks":
    _project_root = _project_root.parent
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from PIL import Image

print("Libraries loaded.")


## Experiment parameters

Set paths and options below. Use either a path relative to the project root or an absolute path.


In [ ]:
# Source media (relative to project root or absolute)
SOURCE_IMAGE = "data/sample_image.jpg"   # image for side-by-side inference
SOURCE_VIDEO = "data/sample_walking.mp4" # video for 5s GIFs

# Model weights: YOLO pose .pt (HALPE 26, CMU 6, or COCO 17 keypoints)
MODEL_WEIGHTS = "yolov8n-pose.pt"  # or e.g. "models/foot_pose/best.pt", "models/yolo_lower/best.pt"

# Output
OUTPUT_DIR = Path("notebooks/output")   # where to save GIFs (and optional image)
GIF_DURATION_SEC = 5                    # length of exported video GIFs in seconds

# Inference options
CONF_THRESHOLD = 0.25                   # detection/keypoint confidence
DEVICE = "auto"                         # "cpu", "0", "cuda:0", "auto", etc.

# Resolve paths from project root
def _path(p):
    s = Path(p)
    if not s.is_absolute():
        s = _project_root / s
    return s

SOURCE_IMAGE = _path(SOURCE_IMAGE)
SOURCE_VIDEO = _path(SOURCE_VIDEO)
# Resolve model path from project root only if it looks like a path (e.g. models/foot_pose/best.pt)
if not Path(MODEL_WEIGHTS).is_absolute() and ("/" in MODEL_WEIGHTS or "\\" in MODEL_WEIGHTS):
    MODEL_WEIGHTS = str(_path(MODEL_WEIGHTS))
OUTPUT_DIR = _path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Parameters set.")


In [ ]:
# Load YOLO pose model (downloads if using a preset name like yolov8n-pose.pt)
model = YOLO(MODEL_WEIGHTS)
print(f"Model loaded: {MODEL_WEIGHTS}")


## Image inference: side-by-side original vs annotated

Run pose on a single image and display original and annotated versions next to each other.

In [ ]:
# --- Image inference: side-by-side original vs annotated ---
if not SOURCE_IMAGE.exists():
    print(f"Image not found: {SOURCE_IMAGE} — set SOURCE_IMAGE in the parameters cell.")
else:
    img_bgr = cv2.imread(str(SOURCE_IMAGE))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    results = model(img_bgr, verbose=False, conf=CONF_THRESHOLD, device=DEVICE)
    annotated_bgr = results[0].plot()  # Ultralytics in-place style plot (BGR)
    annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    ax1.imshow(img_rgb)
    ax1.set_title("Original")
    ax1.axis("off")
    ax2.imshow(annotated_rgb)
    ax2.set_title("With pose")
    ax2.axis("off")
    plt.tight_layout()
    plt.show()


## Video inference: 5-second GIFs

Run pose on the video and export two GIFs (original and annotated), each 5 seconds long.


In [ ]:
# --- Video inference: collect 5 seconds of frames, then write two GIFs ---
if not SOURCE_VIDEO.exists():
    print(f"Video not found: {SOURCE_VIDEO} — set SOURCE_VIDEO in the parameters cell.")
else:
    cap = cv2.VideoCapture(str(SOURCE_VIDEO))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    n_frames = int(GIF_DURATION_SEC * fps)
    frames_original = []
    frames_annotated = []
    for _ in range(n_frames):
        ret, frame = cap.read()
        if not ret:
            break
        frames_original.append(frame.copy())
        results = model(frame, verbose=False, conf=CONF_THRESHOLD, device=DEVICE)
        ann = results[0].plot()
        frames_annotated.append(ann)
    cap.release()

    if not frames_original:
        print("No frames read from video.")
    else:
        # Convert BGR to RGB for GIF (PIL uses RGB)
        def bgr_to_rgb_list(frames):
            return [cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in frames]

        duration_ms = int(1000.0 / fps)  # ms per frame for GIF
        gif_original = OUTPUT_DIR / "video_original_5s.gif"
        gif_annotated = OUTPUT_DIR / "video_annotated_5s.gif"

        imgs_orig = [Image.fromarray(f) for f in bgr_to_rgb_list(frames_original)]
        imgs_orig[0].save(gif_original, save_all=True, append_images=imgs_orig[1:], duration=duration_ms, loop=0)
        imgs_ann = [Image.fromarray(f) for f in bgr_to_rgb_list(frames_annotated)]
        imgs_ann[0].save(gif_annotated, save_all=True, append_images=imgs_ann[1:], duration=duration_ms, loop=0)

        print(f"GIFs saved ({len(frames_original)} frames, {GIF_DURATION_SEC}s at {fps:.1f} FPS):")
        print(f"  Original:  {gif_original}")
        print(f"  Annotated: {gif_annotated}")


In [ ]:
# Optional: display the annotated GIF in the notebook (if in Jupyter)
# from IPython.display import Image as IPImage
# display(IPImage(filename=str(OUTPUT_DIR / "video_annotated_5s.gif")))


# End of example inference. Adjust SOURCE_IMAGE, SOURCE_VIDEO, and MODEL_WEIGHTS in the parameters cell and re-run.

## Next steps

- Place a sample image at `data/sample_image.jpg` and a video at `data/sample_walking.mp4`, or set `SOURCE_IMAGE` / `SOURCE_VIDEO` in the parameters cell.
- Use a custom pose model (e.g. `models/foot_pose/best.pt` or `models/yolo_lower/best.pt`) by setting `MODEL_WEIGHTS`.
- Uncomment the optional cell above to display the annotated GIF inside the notebook.
